This notebook was used to generate the UN General Debate corpus chunked dataset available at

https://huggingface.co/datasets/kalebr/un-general-debate-corpus-chunked

We begin by fetching the original dataset, which was prepared by Alexander Baturo, Niheer Dasandi, and Slava Mikhaylov, and is presented in the paper "Understanding State Preferences With Text As Data: Introducing the UN General Debate Corpus" Research & Politics, 2017.

In [ ]:
!curl -L -o un-general-debates.zip\
  https://www.kaggle.com/api/v1/datasets/download/unitednations/un-general-debates

!unzip un-general-debates.zip

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv('un-general-debates.csv')
df.head()

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device("cuda")
else:
    print("GPU not available, using CPU")
    device = torch.device("cpu")

Here we use the Chonkie library's semantic chunker to chunk each speech.

In [ ]:

from tqdm import tqdm
import pandas as pd
from chonkie import SemanticChunker, AutoEmbeddings
import numpy as np

BATCH_SIZE = 256  # Adjust depending on GPU memory

all_chunks = []

# Stage 1: Fast embedding for chunking
fast_embedder = AutoEmbeddings.get_embeddings("all-MiniLM-L6-v2")
chunker = SemanticChunker(embedding_model=fast_embedder, threshold=0.8, chunk_size=512)

# Stage 2: High-quality embeddings
hq_embedder = AutoEmbeddings.get_embeddings("all-mpnet-base-v2")

# Step 1: Chunk all documents first
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Chunking docs"):
    text = row['text']
    chunks = chunker.chunk(text)

    for chunk in chunks:
        # Store chunk text + metadata for batching later
        if chunk.token_count > 100:
            all_chunks.append({
                'original_index': idx,
                'original_metadata': row.to_dict(),
                'chunk_text': chunk.text,
                'token_count': chunk.token_count
            })




In [ ]:
# Step 2: Batch embed all chunks
chunk_texts = [c['chunk_text'] for c in all_chunks]

embeddings = []
for i in tqdm(range(0, len(chunk_texts), BATCH_SIZE), desc="Computing embeddings"):
    batch_texts = chunk_texts[i:i+BATCH_SIZE]
    batch_embeddings = hq_embedder.embed(batch_texts)  # This should handle batches
    embeddings.extend(batch_embeddings)

# Step 3: Attach embeddings back to all_chunks
for i, emb in enumerate(embeddings):
    all_chunks[i]['embedding'] = np.array(emb, dtype=np.float32)

# Step 4: Convert to DataFrame
chunk_df = pd.DataFrame(all_chunks)

for key in chunk_df['original_metadata'].values[0].keys():
  chunk_df[key] = chunk_df['original_metadata'].apply(lambda x: x[key])
chunk_df = chunk_df.drop(columns=['original_metadata'])
chunk_df = chunk_df.drop(columns=['text'])
chunk_df.head()

print(chunk_df.head())

In [ ]:
from umap import UMAP

reducer = UMAP(verbose=True)
reduced_vectors = reducer.fit_transform(np.stack(chunk_df['embedding'].values))
chunk_df['reduced'] = reduced_vectors.tolist()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from vectorizers.transformers  import InformationWeightTransformer

# Step 1: Convert text to term frequency counts
count_vectorizer = CountVectorizer(max_features=1000)
count_matrix = count_vectorizer.fit_transform(chunk_df['chunk_text'],y=chunk_df['original_index'])

# Step 2: Apply TF-IDF transformation to the count matrix
#fidf_transformer = TfidfTransformer()
iw_transformer = InformationWeightTransformer()
iw_matrix = iw_transformer.fit_transform(count_matrix)

# Sum TF-IDF scores per chunk (informativeness score)
chunk_df['information_weight'] = iw_matrix.sum(axis=1).A1

chunk_df.head()

In [ ]:
chunk_df.to_parquet("ungdc-all-chunked.parquet")